# 07 — Behavioral Momentum & Amelioration

## Part A — Behavioral Momentum

**Key measure**: resistance-to-change ratio `r = responses_extinction / responses_baseline`

Expected: `r_rich > r_lean` (rich reinforcement history confers greater momentum).

Model (Nevin, 1992): `log(B_d / B_0) = −b · d · R^(−a)`

- `B_d` = responses under disruption, `B_0` = baseline responses
- `R` = baseline reinforcement rate, `d` = disruptor magnitude
- `a` = sensitivity of momentum to reinforcement rate
- `b` = disruptor efficacy

## Part B — Amelioration

**Molar**: GML fit comparing FR-FR vs. VI-VI sensitivity.

**Local-rate (melioration test)**:
`P(switch to rich | local_rate_rich > local_rate_lean)` vs.
`P(switch to rich | local_rate_rich < local_rate_lean)`

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load(name):
    p = PROC_DIR / f'{name}.parquet'
    return pd.read_parquet(p) if p.exists() else pd.DataFrame()


---
# Part A — Behavioral Momentum


## A1. Response rates — baseline vs. disruption/extinction


In [ ]:
mom = load('behavioral_momentum')

if not mom.empty:
    # Phase is stored in condition field: 'baseline' or 'disruption'
    # Track: RESP_RICH vs RESP_LEAN event codes
    resp_codes = {'RESP_RICH': 'rich', 'RESP_LEAN': 'lean',
                  'PERSIST': 'persist'}

    rate_rows = []
    for (pid, phase, sid), grp in mom.groupby(
            ['participant_id', 'condition', 'session_id']):
        for code, track in resp_codes.items():
            n = (grp['event_code'] == code).sum()
            dur = grp['elapsed_s'].max()
            rate_rows.append({
                'participant_id': pid,
                'phase':          phase,
                'track':          track,
                'n_responses':    n,
                'duration_s':     dur,
                'rate_per_min':   n / (dur / 60) if dur > 0 else 0,
            })

    rate_df = pd.DataFrame(rate_rows)
    display(rate_df.groupby(['track', 'phase'])
                   .agg(mean_rate=('rate_per_min', 'mean'),
                        sem_rate=('rate_per_min', 'sem'),
                        n=('rate_per_min', 'count'))
                   .round(3))


## A2. Resistance-to-change ratio


In [ ]:
if 'rate_df' in dir() and not rate_df.empty:
    # Pivot: baseline and disruption rates per participant × track
    pivot = rate_df[rate_df['track'].isin(['rich', 'lean'])].pivot_table(
        index=['participant_id', 'track'],
        columns='phase',
        values='rate_per_min'
    )
    pivot.columns.name = None

    if 'baseline' in pivot.columns and 'disruption' in pivot.columns:
        pivot['resistance_ratio'] = pivot['disruption'] / pivot['baseline'].replace(0, np.nan)
        pivot['log_ratio']        = np.log(pivot['resistance_ratio'].replace(0, np.nan))
        display(pivot.round(3))

        # Bar chart: resistance ratio by track
        rich = pivot.xs('rich', level='track')['resistance_ratio'].dropna()
        lean = pivot.xs('lean', level='track')['resistance_ratio'].dropna()

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(['Rich (CRF)', 'Lean (FR10)'],
               [rich.mean(), lean.mean()],
               yerr=[rich.sem(), lean.sem()],
               capsize=5, color=['#2a9d8f', '#e63946'], alpha=0.85)
        ax.axhline(1.0, ls='--', color='gray', lw=1, label='No change (ratio = 1)')
        ax.set_ylabel('Resistance-to-change ratio\n(disruption rate / baseline rate)')
        ax.set_title('Behavioral Momentum — Resistance to Change')
        ax.legend(fontsize=9)
        fig.savefig(FIG_DIR / 'momentum_resistance_ratio.png', bbox_inches='tight')
        plt.show()

        if len(rich) >= 2 and len(lean) >= 2:
            t, p = stats.ttest_rel(rich.values[:min(len(rich),len(lean))],
                                    lean.values[:min(len(rich),len(lean))])
            print(f'\nPaired t-test (rich vs lean resistance): t = {t:.3f}, p = {p:.4f}')


## A3. Nevin (1992) momentum model fit

`log(B_d / B_0) = −b · d · R^(−a)`

Where `R` = reinforcement rate (rich=30/session, lean=3/session),
`d` = disruptor magnitude (fixed at 1 for free-SR+ disruptor).


In [ ]:
if 'pivot' in dir() and 'resistance_ratio' in pivot.columns:
    # Reinforcement rates from spec
    R_vals = {'rich': 30, 'lean': 3}   # reinforcers per session
    d = 1.0                             # disruptor magnitude (normalized)

    model_rows = []
    for pid in pivot.index.get_level_values('participant_id').unique():
        pts = []
        for track in ['rich', 'lean']:
            try:
                ratio = pivot.loc[(pid, track), 'resistance_ratio']
                if pd.notna(ratio) and ratio > 0:
                    pts.append((R_vals[track], np.log(ratio)))
            except KeyError:
                pass

        if len(pts) < 2:
            continue

        R_arr = np.array([p[0] for p in pts])
        y_arr = np.array([p[1] for p in pts])  # log(B_d/B_0)

        # Fit: y = -b * d * R^(-a)  →  minimize SSE over a, b
        def nevin_nll(params):
            a, b = params
            pred = -b * d * R_arr**(-a)
            return np.sum((y_arr - pred)**2)

        res = minimize(nevin_nll, x0=[1.0, 1.0],
                       bounds=[(0.01, 5), (0.01, 10)],
                       method='L-BFGS-B')
        a_hat, b_hat = res.x
        model_rows.append({
            'participant_id': pid,
            'sensitivity_a':  round(a_hat, 4),
            'disruptor_efficacy_b': round(b_hat, 4),
            'sse': round(res.fun, 6)
        })

    if model_rows:
        mod_df = pd.DataFrame(model_rows)
        print('=== Nevin (1992) Momentum Model Estimates ===')
        display(mod_df)
        print('\na = sensitivity of momentum to reinforcement rate')
        print('b = disruptor efficacy (how much the disruptor disrupts)')


---
# Part B — Amelioration


## B1. Molar allocation — FR-FR vs. VI-VI


In [ ]:
aml = load('amelioration')

SCHED_LABELS = {'FR_FR': 'Concurrent FR-FR (3:1)', 'VI_VI': 'Concurrent VI-VI (3:1)'}
SCHED_RATIO  = 3.0   # richness ratio left:right

if not aml.empty:
    # condition column stores 'FR_FR' or 'VI_VI'
    choices = aml[aml['event_code'].isin(['LEFT', 'RIGHT', 'LEFT_SR', 'RIGHT_SR'])]

    molar_rows = []
    for (pid, cond), grp in choices.groupby(['participant_id', 'condition']):
        B1 = (grp['event_code'] == 'LEFT').sum()
        B2 = (grp['event_code'] == 'RIGHT').sum()
        R1 = (grp['event_code'] == 'LEFT_SR').sum()
        R2 = (grp['event_code'] == 'RIGHT_SR').sum()
        total_B = B1 + B2
        total_R = R1 + R2
        molar_rows.append({
            'participant_id': pid,
            'condition': cond,
            'B1': B1, 'B2': B2, 'R1': R1, 'R2': R2,
            'B1_prop': B1 / max(total_B, 1),
            'R1_prop': R1 / max(total_R, 1),
            'scheduled_R1_prop': SCHED_RATIO / (SCHED_RATIO + 1),
        })

    molar_df = pd.DataFrame(molar_rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    for ax, cond in zip(axes, ['FR_FR', 'VI_VI']):
        cdf = molar_df[molar_df['condition'] == cond]
        ax.scatter(cdf['R1_prop'], cdf['B1_prop'], s=80, zorder=3,
                   color='#457b9d', label='Obtained')
        ax.plot([0,1],[0,1], 'k--', lw=1, label='Strict matching')
        ax.axhline(cdf['B1_prop'].mean(), ls=':', color='#457b9d', lw=1.5,
                   label=f'Mean B1 prop = {cdf["B1_prop"].mean():.2f}')
        ax.axvline(SCHED_RATIO/(SCHED_RATIO+1), ls=':', color='orange', lw=1.5,
                   label=f'Scheduled R1 prop = {SCHED_RATIO/(SCHED_RATIO+1):.2f}')
        ax.set_title(SCHED_LABELS.get(cond, cond))
        ax.set_xlabel('Proportion reinforcers from left')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)
    axes[0].set_ylabel('Proportion choices to left')
    fig.suptitle('Amelioration — Molar Allocation by Schedule Type', fontsize=13)
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'amelioration_molar_allocation.png', bbox_inches='tight')
    plt.show()

    # Fit GML per condition
    valid = molar_df[(molar_df['B1']>0)&(molar_df['B2']>0)&
                     (molar_df['R1']>0)&(molar_df['R2']>0)].copy()
    valid['log_B'] = np.log(valid['B1']/valid['B2'])
    valid['log_R'] = np.log(valid['R1']/valid['R2'])

    for cond, cgrp in valid.groupby('condition'):
        if len(cgrp) < 2:
            continue
        slope, intercept, r, p, _ = stats.linregress(cgrp['log_R'], cgrp['log_B'])
        print(f"{SCHED_LABELS.get(cond, cond)}:  a = {slope:.3f}, b = {np.exp(intercept):.3f}, R² = {r**2:.3f}")


## B2. Local-rate melioration test

Test: `P(switch to left | local_rate_left > local_rate_right)` >
`P(switch to left | local_rate_left < local_rate_right)`

Local rate = reinforcers received from each option in a rolling 5-choice window.


In [ ]:
if not aml.empty:
    WINDOW = 5  # rolling window size

    melio_rows = []
    for (pid, cond), grp in aml.groupby(['participant_id', 'condition']):
        grp = grp.sort_values('elapsed_s').reset_index(drop=True)

        # Build choice sequence
        seq = []
        for _, row in grp.iterrows():
            if row['event_code'] == 'LEFT':
                seq.append({'choice': 'L', 'sr': 0})
            elif row['event_code'] == 'RIGHT':
                seq.append({'choice': 'R', 'sr': 0})
            elif row['event_code'] == 'LEFT_SR' and seq:
                seq[-1]['sr'] = 1
            elif row['event_code'] == 'RIGHT_SR' and seq:
                seq[-1]['sr'] = 1
        if len(seq) < WINDOW + 1:
            continue

        seq_df = pd.DataFrame(seq)
        seq_df['left_sr']  = (seq_df['choice'] == 'L') * seq_df['sr']
        seq_df['right_sr'] = (seq_df['choice'] == 'R') * seq_df['sr']

        for i in range(WINDOW, len(seq_df)):
            window = seq_df.iloc[i-WINDOW:i]
            local_L = window['left_sr'].sum()  / max((window['choice']=='L').sum(), 1)
            local_R = window['right_sr'].sum() / max((window['choice']=='R').sum(), 1)
            next_choice = seq_df.iloc[i]['choice']
            switch_to_L = int(next_choice == 'L')
            rich_is_L   = int(local_L > local_R)
            melio_rows.append({
                'participant_id': pid,
                'condition':      cond,
                'local_L':        local_L,
                'local_R':        local_R,
                'rich_is_left':   rich_is_L,
                'switch_to_left': switch_to_L,
            })

    if melio_rows:
        ml_df = pd.DataFrame(melio_rows)

        # Melioration test
        print('=== Melioration Test ===')
        print('P(choose left | local rate left > right) vs.')
        print('P(choose left | local rate right > left)\n')

        for cond in ml_df['condition'].unique():
            cdf = ml_df[ml_df['condition'] == cond]
            p_switch_when_L_rich = cdf[cdf['rich_is_left']==1]['switch_to_left'].mean()
            p_switch_when_R_rich = cdf[cdf['rich_is_left']==0]['switch_to_left'].mean()
            print(f"{SCHED_LABELS.get(cond, cond)}")
            print(f"  P(left | L richer) = {p_switch_when_L_rich:.3f}")
            print(f"  P(left | R richer) = {p_switch_when_R_rich:.3f}")
            direction = 'supports' if p_switch_when_L_rich > p_switch_when_R_rich else 'does NOT support'
            print(f"  → {direction} melioration\n")

        # Run-length analysis
        print('=== Run-Length Analysis ===')
        for (pid, cond), grp in aml.groupby(['participant_id','condition']):
            choices_only = grp[grp['event_code'].isin(['LEFT','RIGHT'])]['event_code'].tolist()
            runs = []
            cur_run = 1
            for i in range(1, len(choices_only)):
                if choices_only[i] == choices_only[i-1]:
                    cur_run += 1
                else:
                    runs.append(cur_run)
                    cur_run = 1
            runs.append(cur_run)
            print(f"  {pid} / {cond}: mean run = {np.mean(runs):.2f}, "
                  f"max run = {max(runs)}, exclusive runs (≥10) = {sum(r>=10 for r in runs)}")


## B3. Local rate over trial sequence


In [ ]:
if 'ml_df' in dir() and not ml_df.empty:
    for cond in ml_df['condition'].unique():
        cdf = ml_df[ml_df['condition'] == cond].reset_index(drop=True)
        if cdf.empty:
            continue

        fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

        for pid, pgrp in cdf.groupby('participant_id'):
            x = np.arange(len(pgrp))
            axes[0].plot(x, pgrp['local_L'], label=f'{pid} left',  alpha=0.7)
            axes[0].plot(x, pgrp['local_R'], label=f'{pid} right', alpha=0.7, ls='--')
            axes[1].plot(x, pgrp['switch_to_left'], alpha=0.5, label=pid)

        axes[0].set_ylabel('Local reinforcement rate')
        axes[0].set_title(f'{SCHED_LABELS.get(cond, cond)} — Local rates (L vs R, rolling {WINDOW}-choice window)')
        axes[0].legend(fontsize=7, ncol=2)

        axes[1].set_ylabel('Chose left (1=yes, 0=no)')
        axes[1].set_xlabel('Trial (post-window)')
        axes[1].set_title('Choice sequence')
        axes[1].legend(fontsize=7)

        plt.tight_layout()
        fig.savefig(FIG_DIR / f'amelioration_local_rate_{cond}.png', bbox_inches='tight')
        plt.show()
